In [26]:
import pandas as pd
import re

In [27]:
def conv(s):
    s = s[1:-1]
    dic = {}

    positions=[match.end() for match in re.finditer(r'\b(?:positive|negative|neutral)\b,', s, flags=re.IGNORECASE)]
    for ss in [s[i:j] for i, j in zip([0] + positions, positions + [None])]:
        tmp = ss.split(':')
        try:
            dic[tmp[0]] = tmp[1].replace(",", "")
        except:
            continue

    return dic

def prepros(df):
    df.sentiments = df.sentiments.str.replace('[\n"\']', '', regex=True)
    df.sentiments = df.sentiments.str.replace(r'{\s+', '{', regex=True)
    df.sentiments = df.sentiments.str.replace(r'\s+}', '}', regex=True)
    df.sentiments = df.sentiments.str.replace(r':\s+', ':', regex=True)
    df.sentiments = df.sentiments.str.replace(r',\s+', ',', regex=True)
    df.sentiments = df.sentiments.apply(conv)

    return df

def top100(df, mapping):
    top = list(mapping.index)
    top_dic = []
    for dic in df.sentiments:
        tmp = {}
        for k in dic:
            if k in top:
                tmp[k] = dic[k]

        top_dic.append(tmp)

    df.sentiments = top_dic           

    return df

In [28]:
files_in = ['altnews', 'boomlive', 'opindia']
files_usa = ['checkyourfact', 'politifact', 'snopes']

All = pd.DataFrame() 

for f in files_in+files_usa:
    tmp_sent = pd.read_excel('../RQ2/Topic Sentiment Data/'+f+'.xlsx')
    tmp_top = pd.read_excel('../RQ2/Top Topics/'+f+'.xlsx')
    
    tmp_top = tmp_top[tmp_top.include==1]
    tmp_top.index = list(tmp_top.ent)

    tmp_sent = prepros(tmp_sent)

    #print(tmp_sent.sentiments)

    tmp_sent = top100(tmp_sent, tmp_top)
    tmp_sent.insert(1, "media_name", [f]*len(tmp_sent), True)
    #print(tmp_sent.sentiments)    
    
    All = pd.concat([All, tmp_sent])

All

,links,media_name,date_year,date_month,date_day,title,text,sentiments
0,https://www.altnews.in/fictitious-politician-a...,altnews,2020,January,15,Fictitious politician ‘Anil Upadhyaya’ makes a...,A video of a politician is being shared in wit...,"{'bjp': 'negative', 'congress': 'neutral'}"
1,https://www.altnews.in/pm-modis-speech-on-caa-...,altnews,2019,December,24,PM Modi’s speech on CAA/NRC: A combination of ...,The BJP’s election campaign for the upcoming p...,"{'narendra modi': 'negative', 'bjp': 'negative..."
2,https://www.altnews.in/movie-still-shared-as-i...,altnews,2019,November,26,Movie still shared as image of 26/11 martyred ...,A photograph of a bloodied policeman is making...,"{'narendra modi': 'positive', 'mahesh vikram h..."
3,https://www.altnews.in/multiple-fake-accounts-...,altnews,2019,August,21,Multiple fake accounts impersonating Indian/Pa...,"On August 19, Republic TV Consulting Editor Ma...",{}
4,https://www.altnews.in/old-video-from-gujarat-...,altnews,2019,May,7,Old video from Gujarat used to falsely claim B...,“मुसलमानों को वोट देने से रोक रहे है मोदी सरका...,"{'narendra modi': 'negative', 'bjp': 'negative..."
...,...,...,...,...,...,...,...,...
9631,https://www.snopes.com/fact-check/trump-inappr...,snopes,2018,January,2,Did Donald Trump Tweet About Daughter Tiffany'...,Claim: President Donald Trump posted a sugges...,{'president donald trump': 'neutral'}
9632,https://www.snopes.com/fact-check/us-returning...,snopes,2018,January,2,Is the United States Returning Puerto Rico to ...,Claim: The United States government has secre...,{'donald trump': 'negative'}
9633,https://www.snopes.com/fact-check/melania-trum...,snopes,2018,January,2,Did Melania Trump Wear Shower Curtains?,Claim: The pattern on Melania Trump's New Yea...,"{'melania trump': 'positive', 'donald trump': ..."
9634,https://www.snopes.com/fact-check/delta-force-...,snopes,2018,January,2,Did Delta Force Operators Raid an 'Obama Stron...,Claim: Delta Force operators raided President...,"{'president trump': 'neutral', 'vladimir putin..."


In [29]:
All = All[All.sentiments.apply(lambda x: len(x)>0)]

All

,links,media_name,date_year,date_month,date_day,title,text,sentiments
0,https://www.altnews.in/fictitious-politician-a...,altnews,2020,January,15,Fictitious politician ‘Anil Upadhyaya’ makes a...,A video of a politician is being shared in wit...,"{'bjp': 'negative', 'congress': 'neutral'}"
1,https://www.altnews.in/pm-modis-speech-on-caa-...,altnews,2019,December,24,PM Modi’s speech on CAA/NRC: A combination of ...,The BJP’s election campaign for the upcoming p...,"{'narendra modi': 'negative', 'bjp': 'negative..."
2,https://www.altnews.in/movie-still-shared-as-i...,altnews,2019,November,26,Movie still shared as image of 26/11 martyred ...,A photograph of a bloodied policeman is making...,"{'narendra modi': 'positive', 'mahesh vikram h..."
4,https://www.altnews.in/old-video-from-gujarat-...,altnews,2019,May,7,Old video from Gujarat used to falsely claim B...,“मुसलमानों को वोट देने से रोक रहे है मोदी सरका...,"{'narendra modi': 'negative', 'bjp': 'negative..."
7,https://www.altnews.in/viral-video-claims-a-te...,altnews,2018,September,1,Viral video claims a terrorist was arrested at...,“जयपुर एयरपोर्ट पर आंतकवादी पकड़ा गया” (A terro...,{'modi government': 'negative'}
...,...,...,...,...,...,...,...,...
9631,https://www.snopes.com/fact-check/trump-inappr...,snopes,2018,January,2,Did Donald Trump Tweet About Daughter Tiffany'...,Claim: President Donald Trump posted a sugges...,{'president donald trump': 'neutral'}
9632,https://www.snopes.com/fact-check/us-returning...,snopes,2018,January,2,Is the United States Returning Puerto Rico to ...,Claim: The United States government has secre...,{'donald trump': 'negative'}
9633,https://www.snopes.com/fact-check/melania-trum...,snopes,2018,January,2,Did Melania Trump Wear Shower Curtains?,Claim: The pattern on Melania Trump's New Yea...,"{'melania trump': 'positive', 'donald trump': ..."
9634,https://www.snopes.com/fact-check/delta-force-...,snopes,2018,January,2,Did Delta Force Operators Raid an 'Obama Stron...,Claim: Delta Force operators raided President...,"{'president trump': 'neutral', 'vladimir putin..."


In [30]:
All = All.sample(frac=1, random_state=42)
All = All.sample(n=200, random_state=42)

#All.to_excel('HE_sentiment.xlsx', index=False)

All.media_name.value_counts()

media_name
politifact       71
snopes           38
checkyourfact    31
boomlive         31
altnews          15
opindia          14
Name: count, dtype: int64